In [ ]:
import json
import os
import glob
from datetime import datetime
import random

def folder_sp3_to_czml(folder_path, czml_filename):
    czml = [{"id": "document", "name": "Konstelacja GPS z folderu SP3", "version": "1.0"}]
    satellites_data = {}
    base_time = None
    base_epoch_iso = ""
    last_time = None  

    search_pattern = os.path.join(folder_path, "*.sp3")
    file_list = glob.glob(search_pattern)

    if not file_list:
        print(f"Błąd: Nie znaleziono plików SP3 w folderze: {folder_path}")
        return

    file_list.sort()
    
    for sp3_filename in file_list:
        with open(sp3_filename, 'r') as file:
            current_time = None
            for line in file:
                if line.startswith('* '):
                    parts = line.split()
                    current_time = datetime(int(parts[1]), int(parts[2]), int(parts[3]), int(parts[4]), int(parts[5]), int(float(parts[6])))
                    if base_time is None:
                        base_time = current_time
                        base_epoch_iso = base_time.isoformat() + "Z"
                    
                    last_time = current_time  

                elif line.startswith('PG'):
                    parts = line.split()
                    sat_id = parts[0]
                    if sat_id not in satellites_data:
                        satellites_data[sat_id] = []
                    if current_time and base_time:
                        time_offset = (current_time - base_time).total_seconds()
                        satellites_data[sat_id].extend([time_offset, float(parts[1]) * 1000, float(parts[2]) * 1000, float(parts[3]) * 1000])


    if base_time and last_time:
        czml[0]["clock"] = {
            "interval": f"{base_epoch_iso}/{last_time.isoformat()}Z",
            "currentTime": base_epoch_iso,
            "multiplier": 300,
            "range": "LOOP_STOP"
        }


    for sat_id, cartesian_data in satellites_data.items():
        r, g, b = random.randint(100, 255), random.randint(100, 255), random.randint(100, 255)
        czml.append({
            "id": sat_id,
            "name": f"GPS {sat_id}",
            "path": {"show": [{"boolean": True}], "width": [{"number": 1.5}], "material": {"solidColor": {"color": {"rgba": [r, g, b, 255]}}}, "resolution": [{"number": 120}], "leadTime": [{"number": 86400}], "trailTime": [{"number": 86400}]},
            "point": {"color": {"rgba": [255, 255, 255, 255]}, "pixelSize": [{"number": 6}], "outlineColor": {"rgba": [r, g, b, 255]}, "outlineWidth": [{"number": 2}]},
            "position": {"interpolationAlgorithm": "LAGRANGE", "interpolationDegree": 5, "referenceFrame": "FIXED", "epoch": base_epoch_iso, "cartesian": cartesian_data}
        })

    with open(czml_filename, 'w') as outfile:
        json.dump(czml, outfile, indent=4)
        
    print("Ok.")

folder_sp3_to_czml('.', 'wynik_satelity.czml')

Gotowe. Plik wynik_satelity.czml został utworzony.
